# Конспект. Модуль 10: Подбор гиперпараметров — от GridSearch к Optuna

## 1. Зачем это нужно и как это связано с предыдущими модулями

В Модуле 9 мы систематизировали **больше десятка** значимых гиперпараметров бустинга по пяти категориям (сложность дерева, веса листьев, стохастичность, ансамбль, структура). Мы научились **вручную**, по одной категории за раз, «лечить» переобученную модель — полезный навык для диагностики, но **не** полноценная стратегия поиска лучшей комбинации параметров. Этот модуль отвечает на естественный следующий вопрос: **как найти действительно оптимальную комбинацию** из всех этих параметров, а не просто «разумную» вручную подобранную?

Важно отметить принципиальный сдвиг: в Модулях 3–9 мы **оптимизировали параметры самой модели** (веса листьев, структуру дерева) — у нас был градиент, аналитическая формула, чёткая математика (Модуль 6). Здесь мы оптимизируем **гиперпараметры** — величины, которые **нельзя** продифференцировать (нет формулы `∂PR-AUC/∂num_leaves`), и каждая «проверка» одной комбинации требует **полного обучения модели** — дорогостоящей операции, занимающей минуты или часы. Это принципиально другой класс задачи оптимизации: **чёрный ящик (black-box optimization)** с дорогой оценочной функцией.

## 2. Почему GridSearchCV не годится на бустинге: комбинаторный взрыв

### 2.1. Напоминание и точный расчёт

`GridSearchCV` (Неделя 5) вам уже знаком: вы задаёте сетку значений для каждого параметра, и алгоритм перебирает **все возможные комбинации**. Посчитаем реалистичный пример для бустинга, используя таксономию Модуля 9:

| Параметр | Число значений в сетке |
|---|---|
| `num_leaves` | 5 |
| `learning_rate` | 5 |
| `min_data_in_leaf` | 4 |
| `lambda_l1` | 4 |
| `lambda_l2` | 4 |
| `feature_fraction` | 3 |
| `bagging_fraction` | 3 |
| `n_estimators` | 3 |

**Общее число комбинаций:**

In [ ]:
5 × 5 × 4 × 4 × 4 × 3 × 3 × 3 = 43 200

**43 200 полных обучений модели** — даже если одно обучение LightGBM занимает всего 10 секунд на вашем датасете, это `43200 × 10 сек ≈ 120 часов` — пять суток непрерывных вычислений, и это ещё без учёта кросс-валидации (обычно каждая точка сетки проверяется на нескольких фолдах, что умножает время ещё в 3–5 раз). Это и есть **комбинаторный взрыв**: число комбинаций растёт **мультипликативно** с каждым новым параметром — переход от 8 параметров с сеткой из 4 значений к 9 параметрам с той же сеткой не добавляет 12.5% работы, а **умножает** общий объём работы на 4.

### 2.2. Промежуточный шаг: RandomizedSearchCV и удивительный результат Бергстра–Бенджио

Прежде чем переходить к «умному» поиску, стоит вспомнить менее умный, но неожиданно эффективный компромисс — **случайный поиск** (`RandomizedSearchCV`, уже частично знакомый вам инструмент): вместо перебора **всех** точек сетки, мы случайно выбираем `K` комбинаций (скажем, `K=200`) и проверяем только их.

Интуитивно кажется, что это просто «более дешёвая, но более грубая» версия Grid Search. Но есть классический результат (Бергстра и Бенджио, 2012 год): **в большинстве реальных задач подбора гиперпараметров реальное влияние на качество оказывают лишь 1–3 из всех перебираемых параметров** — остальные варьируются в широких пределах почти без эффекта на итоговую метрику. Grid Search **тратит** одинаковый бюджет на все измерения сетки одинаково, включая «неважные» — из-за этого при фиксированном бюджете проверок он **вынужденно** проверяет лишь горстку **различных** значений по-настоящему важных параметров (так как большая часть комбинаций отличается друг от друга только неважными параметрами). Random Search, наоборот, при том же бюджете `K` проверок **гарантированно** пробует `K` **разных** значений по каждому параметру одновременно (так как каждая точка выбирается независимо и случайно по всем осям сразу) — включая по-настоящему важные, получая **более широкое покрытие** именно тех измерений, которые реально определяют результат.

**Численная иллюстрация.** Пусть из 8 параметров реально важны только 2 (скажем, `num_leaves` и `learning_rate`), а остальные 6 не влияют на качество вовсе. При сетке `3` значения на параметр, `Grid Search` берёт `3^8 = 6561` комбинации — но осмысленных, **различных по результату**, среди них — только `3^2=9` (то есть каждая уникальная пара «важных» значений повторяется в среднем `6561/9 ≈ 729` раз с разными, но не имеющими значения комбинациями остальных 6 параметров). `Random Search` с бюджетом всего `100` проверок, напротив, покроет пространство **важных** двух параметров гораздо плотнее и разнообразнее, потратив на порядки меньше вычислений.

**Вывод:** Random Search — уже неплохая, широко используемая база (и часто более эффективна, чем Grid, при равном бюджете), но у неё есть очевидный недостаток — она **не учится** на уже выполненных попытках: 200-я случайная точка выбирается так же «вслепую», как и первая, даже если предыдущие 199 попыток уже явно показали, в каком «районе» пространства параметров результаты лучше. Именно эту проблему решает следующий шаг — байесовская оптимизация.

## 3. Байесовская оптимизация: строим модель «какие параметры хороши»

### 3.1. Общая идея

Вместо того чтобы выбирать следующую точку случайно или по фиксированной сетке, байесовская оптимизация **накапливает опыт**: после каждой проверенной комбинации гиперпараметров (и результата — например, `val PR-AUC`) она обновляет **вероятностную модель** («суррогатную модель», surrogate model) того, как **предположительно** выглядит зависимость качества от гиперпараметров **во всём** пространстве поиска, включая ещё не проверенные области. Основываясь на этой модели, алгоритм выбирает **следующую** точку не случайно, а осмысленно — там, где, по текущей модели, либо (а) ожидается наилучший результат (**эксплуатация**, exploitation), либо (б) неопределённость модели особенно высока — то есть область, о которой мы пока мало знаем и там может скрываться неожиданно хорошая комбинация (**исследование**, exploration). Хороший алгоритм байесовской оптимизации **балансирует** между этими двумя стратегиями.

### 3.2. TPE — алгоритм, который использует Optuna

Классический подход байесовской оптимизации использует Гауссовские процессы как суррогатную модель — математически элегантно, но плохо масштабируется на много измерений и на параметры смешанного типа (числовые + категориальные, как у нас — `num_leaves` целочисленный, `feature_fraction` вещественный, `boosting_type` категориальный). Optuna по умолчанию использует другой алгоритм — **TPE (Tree-structured Parzen Estimator)**, который справляется с этим значительно проще и быстрее.

**Идея TPE пошагово:**

1. После каждой пробы (`trial`) у нас накапливается список пар `(гиперпараметры, результат)`.
2. Выбирается порог `γ` (например, `γ=0.2` — лучшие 20% проб по результату).
3. Все прошлые пробы делятся на две группы: **«хорошие»** (`l`, результат в лучших `γ`) и **«плохие»** (`g`, все остальные).
4. Для каждой группы строится **плотность распределения** значений гиперпараметра — не аналитическая формула, а эмпирическая оценка (по сути, «где чаще встречаются значения этого параметра среди хороших проб, а где — среди плохих»).
5. Следующая точка для проверки выбирается там, где отношение плотностей **максимально**:

In [ ]:
следующая_точка = argmax(x)  l(x) / g(x)

**Интуиция формулы:** мы ищем область значений параметра, которая **часто** встречалась среди хороших результатов (`l(x)` велико), но **редко** — среди плохих (`g(x)` мало). Высокое отношение `l(x)/g(x)` — сильный сигнал «здесь, скорее всего, хорошо».

### 3.3. Численная иллюстрация TPE

Представим, что мы уже провели `10` проб, подбирая только `learning_rate`, и получили следующие результаты (`val_loss`, чем меньше — тем лучше):

| Проба | learning_rate | val_loss |
|---|---|---|
| 1 | 0.01 | 0.50 |
| 2 | 0.05 | 0.30 |
| 3 | 0.10 | 0.25 |
| 4 | 0.15 | 0.28 |
| 5 | 0.20 | 0.35 |
| 6 | 0.30 | 0.45 |
| 7 | 0.40 | 0.55 |
| 8 | 0.50 | 0.60 |
| 9 | 0.02 | 0.48 |
| 10 | 0.08 | 0.27 |

**Сортируем по `val_loss` (по возрастанию, лучшие — первые):** `0.25 (lr=0.10)`, `0.27 (lr=0.08)`, `0.28 (lr=0.15)`, `0.30 (lr=0.05)`, `0.35 (lr=0.20)`, `0.45 (lr=0.30)`, `0.48 (lr=0.02)`, `0.50 (lr=0.01)`, `0.55 (lr=0.40)`, `0.60 (lr=0.50)`.

При `γ=0.3` **«хорошая» группа** (лучшие 3 из 10) — `learning_rate ∈ {0.10, 0.08, 0.15}` — узко сконцентрирована в диапазоне **0.08–0.15**.

**«Плохая» группа** (остальные 7) — `{0.05, 0.20, 0.30, 0.02, 0.01, 0.40, 0.50}` — заметно **более разбросана**, причём в **обе** стороны от «хорошей» зоны: и слишком маленькие значения (`0.01, 0.02, 0.05` — вероятно, модель не успевает сойтись за отведённое число итераций, вспомните Модуль 5), и слишком большие (`0.30–0.50` — вероятно, слишком крупный, нестабильный шаг).

**Вывод, к которому придёт TPE:** плотность `l(x)` резко сконцентрирована вокруг `0.08–0.15`, а `g(x)` — размазана широко по краям этого диапазона с обеих сторон. Отношение `l(x)/g(x)` будет максимально **внутри** узкой хорошей зоны — алгоритм предложит на следующем шаге попробовать что-то вроде `learning_rate≈0.11–0.12`, уточняя именно этот перспективный диапазон, а не продолжая случайно бросать точки по всему исходному пространству от 0.01 до 0.50, как это делал бы Random Search.

**Именно это и есть принципиальное отличие от случайного перебора (ответ на первый чек-поинт вопрос):** байесовская оптимизация **обучается** на всех предыдущих пробах и целенаправленно сужает область поиска к перспективным зонам, тогда как случайный перебор каждый раз выбирает точку **независимо** от истории — 100-я случайная проба ничем не «умнее» первой.

*(Историческая справка для полноты картины: в оригинальной статье про TPE показано, что максимизация `l(x)/g(x)` математически связана с максимизацией так называемого **Expected Improvement** — величины из классической Байесовской оптимизации, показывающей ожидаемое улучшение результата в данной точке относительно текущего лучшего найденного значения. Разные формулировки, но по сути — одна и та же цель: искать там, где высок шанс превзойти уже найденный рекорд.)*

## 4. Практическая стратегия тюнинга: не всё сразу

### 4.1. Может ли Optuna искать сразу по всем параметрам совместно?

Формально — да, и это часто **лучше**, чем последовательный поиск по категориям (Модуль 9): совместный поиск способен найти комбинации, где, например, оптимальное `lambda_l2` **зависит** от выбранного `num_leaves` (эффекты взаимодействия между параметрами, которые последовательный, «по одной категории», подход в принципе не может обнаружить — ведь он фиксирует одни параметры, пока подбирает другие).

### 4.2. Почему на практике всё же часто используют поэтапный подход

Проблема совместного поиска **по всем** параметрам сразу, с самого начала и с широкими, «наугад» заданными границами — **объём пространства поиска** растёт экспоненциально с числом измерений, и TPE (как и любой байесовский метод) нуждается в **достаточном числе проб**, чтобы построить содержательные `l(x)` и `g(x)` в каждой значимой области **этого** большого пространства. Чем шире и многомернее исходное пространство, тем больше проб нужно, прежде чем алгоритм начнёт находить действительно полезные закономерности, а не блуждать почти случайно на первых десятках итераций.

**Практический компромисс, который используют на практике:**
1. Сначала — **грубый** поиск по параметрам с наибольшим ожидаемым влиянием (Категория A из Модуля 9: `num_leaves`/`max_depth`, плюс `learning_rate`) с относительно небольшим числом проб, чтобы **сузить разумные границы** для этих параметров.
2. Затем — **более тонкий**, совместный поиск по **оставшимся** параметрам (регуляризация, стохастичность), уже **внутри** найденных на первом шаге разумных диапазонов «крупных» параметров — либо зафиксировав их, либо сузив их диапазон поиска на втором этапе.
3. Даже если в итоге запускается полностью совместный поиск по всем параметрам сразу, **разумные, информированные границы** диапазонов (полученные из интуиции Модуля 9 и/или первого грубого прохода) резко сокращают эффективный объём пространства, который TPE должен исследовать — это не строгая необходимость, а практический способ не тратить бюджет проб на заведомо бессмысленные комбинации (например, незачем гонять `num_leaves` от 2 до 10000, если разумный диапазон для вашей задачи — 20–150).

**Прямой ответ на второй чек-поинт вопрос модуля:** тюнить не всё сразу разумно не потому, что совместный поиск **математически хуже** (напротив, он может найти взаимодействия, недоступные последовательному подходу), а потому что **вычислительный бюджет проб ограничен**, а пространство поиска растёт экспоненциально с числом параметров — сужение диапазонов на основе уже понятой структуры задачи (Модуль 9) делает каждую пробу «дороже» по информативности и позволяет byesian-методу сойтись к хорошему решению за реалистичное число итераций.

## 5. Практика: код

### 5.1. Базовый цикл Optuna

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import average_precision_score

X, y = make_classification(n_samples=40000, n_features=40, n_informative=18,
                            weights=[0.93, 0.07], random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

def objective(trial: optuna.Trial) -> float:
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 15, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 200),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "n_estimators": 300,
        "random_state": 42,
        "verbose": -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    val_pr_auc = average_precision_score(y_val, model.predict_proba(X_val)[:, 1])
    return val_pr_auc

study = optuna.create_study(direction="maximize")  # PR-AUC - чем больше, тем лучше
study.optimize(objective, n_trials=25)

print("Лучшие параметры:", study.best_params)
print("Лучший val PR-AUC:", study.best_value)

**Обратите внимание на `log=True`** у `learning_rate`, `lambda_l1`, `lambda_l2` — это указывает Optuna сэмплировать значения **равномерно в логарифмической шкале**, а не в линейной. Это важно: разница между `learning_rate=0.01` и `0.02` (в 2 раза) содержательно сопоставима с разницей между `0.15` и `0.30` — в линейной шкале эти пары находятся на совершенно разном расстоянии друг от друга, что исказило бы поиск. Логарифмическая шкала для параметров, которые «работают по порядку величины» (шаг обучения, коэффициенты регуляризации) — стандартная практика.

### 5.2. Визуализация истории оптимизации

In [ ]:
import optuna.visualization as vis

fig = vis.plot_optimization_history(study)
fig.show()

fig2 = vis.plot_param_importances(study)
fig2.show()

**Что искать на `plot_optimization_history`:** по горизонтали — номер пробы, по вертикали — достигнутое значение метрики (обычно рисуется текущий лучший найденный результат нарастающим итогом). В отличие от Random Search, где улучшения появлялись бы более-менее равномерно случайно по всей истории, для TPE типична картина: заметный прогресс в первые ~30–40% проб (алгоритм ещё исследует пространство и накапливает статистику для `l(x)`/`g(x)`), затем всё более редкие, но иногда всё ещё случающиеся уточняющие улучшения — TPE постепенно переходит от исследования к эксплуатации найденной хорошей области.

`plot_param_importances` — отдельно полезная функция: показывает, **какие** из перебираемых гиперпараметров реально повлияли на результат в ваших `n_trials` пробах — прямая практическая иллюстрация результата Бергстра–Бенджио из раздела 2.2: скорее всего, вы увидите, что 2–3 параметра доминируют по важности, а остальные вносят минимальный вклад.

### 5.3. Pruning — досрочное прерывание неперспективных проб

Дополнительная оптимизация, которую стоит знать: Optuna умеет прерывать **отдельную** пробу **досрочно**, если по промежуточным результатам (например, после половины запланированных итераций бустинга) видно, что она заведомо хуже уже найденных лучших проб — не тратя оставшееся время на её полное завершение.

In [ ]:
def objective_with_pruning(trial: optuna.Trial) -> float:
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 15, 200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "verbose": -1,
        "random_state": 42,
    }

    pruning_callback = optuna.integration.LightGBMPruningCallback(trial, "average_precision")

    model = lgb.LGBMClassifier(n_estimators=500, **params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="average_precision",
        callbacks=[pruning_callback],
    )
    return average_precision_score(y_val, model.predict_proba(X_val)[:, 1])

study_pruned = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=20)
)
study_pruned.optimize(objective_with_pruning, n_trials=25)

**Идея `MedianPruner`:** после `n_warmup_steps` промежуточных шагов внутри пробы её текущий результат сравнивается с **медианой** промежуточных результатов на этом же шаге среди всех предыдущих проб — если текущая проба заметно хуже медианы, она прерывается. Это концептуальное расширение идеи ранней остановки из Модуля 5 (там мы останавливали **одно** обучение по его собственной val-кривой) на уровень **между** пробами — экономит вычислительный бюджет, позволяя потратить сэкономленное время на больше содержательных проб.

## 6. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Почему GridSearchCV плохо подходит для тюнинга бустинга? | Число комбинаций растёт мультипликативно с числом параметров («комбинаторный взрыв») — с 8+ значимыми гиперпараметрами полный перебор становится вычислительно неподъёмным |
| Почему Random Search часто лучше Grid Search при равном бюджете? | Обычно значимо влияют лишь несколько параметров из многих; Grid тратит бюджет равномерно на все измерения (включая неважные), Random Search за счёт независимого сэмплирования каждой точки естественно даёт более плотное покрытие именно значимых измерений |
| Чем байесовская оптимизация принципиально отличается от случайного перебора? | Она строит вероятностную модель по уже выполненным пробам и выбирает следующую точку осмысленно (баланс exploration/exploitation), а не независимо от истории — случайный перебор «не помнит» и не использует результаты предыдущих попыток |
| Как работает TPE в общих чертах? | Прошлые пробы делятся на «хорошие» и «плохие» по порогу `γ`; строятся плотности распределения гиперпараметров в каждой группе; следующая точка выбирается там, где отношение «хорошей» плотности к «плохой» максимально |
| В каком порядке разумно тюнить параметры бустинга и почему не всё сразу? | Не потому что совместный поиск хуже математически (он может находить взаимодействия параметров), а потому что вычислительный бюджет ограничен, а пространство поиска растёт экспоненциально — практично сначала грубо сузить диапазоны «крупных» параметров (сложность дерева, learning rate), затем искать тоньше внутри суженного пространства |

## 7. Чек-поинт — попробуйте ответить без подсказок

1. Чем байесовская оптимизация принципиально отличается от случайного перебора?
2. В каком порядке разумно тюнить параметры бустинга и почему не все сразу?
3. Почему Random Search при равном бюджете часто превосходит Grid Search, хотя оба формально «не учатся» на предыдущих пробах?
4. Что означает `l(x)/g(x)` в TPE, и почему следующая точка выбирается там, где это отношение максимально?
5. Зачем для `learning_rate` и коэффициентов регуляризации в Optuna используется логарифмическая шкала сэмплирования (`log=True`), а не линейная?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Байесовская оптимизация накапливает информацию обо всех уже выполненных пробах и строит (явную или неявную) вероятностную модель зависимости качества от гиперпараметров, используя её, чтобы **целенаправленно** выбрать следующую точку — либо там, где ожидается лучший результат, либо там, где велика неопределённость (потенциально скрывающая ещё не найденное хорошее решение). Случайный перебор выбирает каждую следующую точку **независимо** от истории — не использует накопленный опыт вообще.

2. Совместный поиск по всем параметрам сразу не хуже математически (он способен находить взаимодействия между параметрами, недоступные последовательному подходу), но пространство поиска растёт экспоненциально с числом измерений, а вычислительный бюджет проб ограничен — байесовскому методу нужно «набрать статистику» в каждой значимой области пространства, и чем оно шире, тем больше проб для этого требуется. Практичнее сначала сузить диапазоны наиболее влиятельных параметров (структура дерева, learning rate) грубым поиском, а затем искать точнее внутри уже суженного, более компактного пространства.

3. Потому что в реальных задачах гиперпараметрического поиска обычно **значимо** влияют на результат лишь несколько измерений из многих. Grid Search тратит одинаковую долю бюджета на все измерения одновременно, включая неважные — большинство его комбинаций отличаются друг от друга только «неважными» осями и по сути повторяют одну и ту же содержательную конфигурацию много раз. Random Search, выбирая каждую точку случайно и независимо по всем осям сразу, гарантированно даёт при том же бюджете **больше различных** значений по-настоящему важных параметров.

4. `l(x)` — эмпирическая плотность значений гиперпараметра среди «хороших» прошлых проб (лучшие по метрике `γ`-доля), `g(x)` — среди «плохих» (остальные). Высокое значение `l(x)/g(x)` в точке `x` означает, что такие значения параметра часто приводили к хорошим результатам и редко — к плохим, то есть это перспективная область для дальнейшего исследования. Выбор следующей точки там, где это отношение максимально, — прямая реализация принципа «учиться на прошлых попытках», центрального для всей байесовской оптимизации.

5. Потому что эти параметры «работают по порядку величины», а не по абсолютной разнице: переход от `learning_rate=0.01` к `0.02` (удвоение) содержательно сопоставим с переходом от `0.15` к `0.30` (тоже удвоение), но в линейной шкале эти пары значений находятся на совершенно разных расстояниях друг от друга (0.01 против 0.15), что исказило бы то, как алгоритм воспринимает «близость» и «удалённость» точек в пространстве поиска. Логарифмическая шкала выравнивает восприятие относительных, а не абсолютных изменений — что соответствует тому, как эти параметры реально влияют на поведение модели.

</details>